## 0. Import libraries

In [1]:
import os
import json
import torch
import wandb
from tqdm import tqdm
from dotenv import load_dotenv
from datasets import * 
from collections import Counter
from pathlib import Path
from prompts import SYSTEM_PROMPT
from collections import defaultdict
from modelscope import snapshot_download, AutoTokenizer
from peft import LoraConfig, TaskType, get_peft_model, PeftModel
from transformers import AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments, Trainer, DataCollatorForSeq2Seq
from transformers.integrations import WandbCallback

load_dotenv()

/venv/qwen_finetune/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


False

In [2]:
wandb.login()

wandb.init(
    project="qwen2.5-7b-ft",
    name='run-1'
)

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: lamyeungkong0108 (lamyeungkong0108-the-hong-kong-university-of-science-and) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


## 1. Dataset Loading

In [2]:
train_dataset_path = "./data/medical.train"
train_json_dataset_path = "./data/medical_train.jsonl"

test_data = "./data/medical.test"
test_json_dataset_path = "./data/medical_test.jsonl"

val_data = "./data/medical.dev"
val_json_dataset_path = "./data/medical_val.jsonl"

In [3]:
def convert_to_json(file_path: str, json_path):
    """
    Converting raw dataset to json format.
    For each data:
    {
        "sentence": the_sentence,
        "entity_info": [
            {
                "entity_text": the_substring_1,
                "entity_label": the_corresponding_label_1         
            },
            {
                "entity_text": the_substring_2,
                "entity_label": the_corresponding_label_2         
            },
            ...
        ]
    }
    """
    current_sub = ""
    current_sentences = ""
    current_dict = []
    current_label = None
    entities_label = []

    json_data = []

    print("Extracting data...")
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            chunk = line.split(" ") # e.g.灵 I-方剂

            if not line: # Empty line  -> previous sentence has finished
                if current_label is not None and current_sub != "":
                    current_dict.append(
                        {
                            "entity_text": current_sub,
                            "entity_label": current_label
                        }
                    )   
                if current_sentences:          
                    json_data.append(
                        {
                            "sentence": current_sentences,
                            "entity_info": current_dict
                        }
                    )        
                current_sub = ""
                current_sentences = ""
                current_dict = []
                current_label = None                

            else: #  O /灵 I-方剂 / 灵 B-方剂
                if chunk[0] == 'O': # e.g.O
                    tag = 'O'
                    label = None
                    char = " "
                elif len(chunk[1]) > 1: # X I-XX / X B-XX, CHUNK[1] = "I-XX"
                    tag, label = chunk[1].split('-')
                    char = chunk[0]
                else: # X O, CHUNK[1] = "O"
                    tag = chunk[1]
                    label = None
                    char = chunk[0]

                current_sentences += char

                if tag == 'I':
                    current_sub += char
                elif tag == 'B':
                    if current_label is not None and current_sub != "":
                        current_dict.append(
                            {
                                "entity_text": current_sub,
                                "entity_label": current_label
                            }
                        )

                    # Assign new label 
                    current_label = label
                    current_sub = char

                    if current_label not in entities_label:
                        entities_label.append(current_label)
                else: # tag == 'O'
                    if current_label is not None and current_sub != "":
                        current_dict.append(
                            {
                                "entity_text": current_sub,
                                "entity_label": current_label
                            }
                        )
                    current_label = None
                    current_sub = ""
    
    open(json_path, 'w').close()
    print(f"Converting {len(json_data)} sentences with their entity information to a jsonal file.")
    with open(json_path, 'a', encoding='utf-8') as f:
        for obj in json_data:
            f.write(json.dumps(obj, ensure_ascii=False) + "\n")
    print("Convert Successfully.")
    return entities_label if entities_label else []
                    

train_json_dataset_path = Path(train_json_dataset_path)
train_json_dataset_path.touch()
train_entities_label = convert_to_json(train_dataset_path, train_json_dataset_path)

test_json_dataset_path = Path(test_json_dataset_path)
test_json_dataset_path.touch()
test_entities_label = convert_to_json(test_data, test_json_dataset_path)

val_json_dataset_path = Path(val_json_dataset_path)
val_json_dataset_path.touch()
val_entities_label = convert_to_json(val_data, val_json_dataset_path)

print(f"All unique entities in training dataset: {train_entities_label}")
print(f"All unique entities in testing dataset: {test_entities_label}")
print(f"All unique entities in valid dataset: {val_entities_label}")

train_entities_label = train_entities_label or []
test_entities_label = test_entities_label or []
val_entities_label = val_entities_label or []
entities_label = list(set(train_entities_label + test_entities_label + val_entities_label))
print(f"All unique entities combined: {entities_label}")

Extracting data...
Converting 5258 sentences with their entity information to a jsonal file.
Convert Successfully.
Extracting data...
Converting 657 sentences with their entity information to a jsonal file.
Convert Successfully.
Extracting data...
Converting 656 sentences with their entity information to a jsonal file.
Convert Successfully.
All unique entities in training dataset: ['临床表现', '中医治疗', '西医诊断', '方剂', '中药', '中医诊断', '西医治疗', '中医证候', '中医治则', '其他治疗']
All unique entities in testing dataset: ['中医诊断', '西医治疗', '西医诊断', '中医治则', '中医治疗', '临床表现', '中医证候', '方剂', '中药', '其他治疗']
All unique entities in valid dataset: ['方剂', '中药', '西医诊断', '中医治疗', '中医证候', '其他治疗', '临床表现', '中医治则', '西医治疗', '中医诊断']
All unique entities combined: ['中医证候', '中医治疗', '其他治疗', '临床表现', '西医诊断', '方剂', '中医治则', '中医诊断', '西医治疗', '中药']


In [4]:
train_dataset = Dataset.from_json(str(train_json_dataset_path))
test_dataset = Dataset.from_json(str(test_json_dataset_path))
val_dataset = Dataset.from_json(str(val_json_dataset_path))

Generating train split: 5258 examples [00:00, 367996.30 examples/s]
Generating train split: 657 examples [00:00, 158225.64 examples/s]
Generating train split: 656 examples [00:00, 171837.59 examples/s]


In [5]:
train_dataset[0:3]

{'sentence': ['现头昏口苦',
  '目的观察复方丁香开胃贴外敷神阙穴治疗慢性心功能不全伴功能性消化不良的临床疗效',
  '舒肝和胃消痞汤；功能性消化不良'],
 'entity_info': [[{'entity_text': '口苦', 'entity_label': '临床表现'}],
  [{'entity_text': '复方丁香开胃贴', 'entity_label': '中医治疗'},
   {'entity_text': '心功能不全伴功能性消化不良', 'entity_label': '西医诊断'}],
  [{'entity_text': '功能性消化不良', 'entity_label': '西医诊断'}]]}

## 2. Model Download 

In [4]:
model_id = "Qwen/Qwen2.5-7B"    

model = snapshot_download(model_id, cache_dir="./qwen")

tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=False, trust_remote_code=False)
tokenizer.pad_token_id = tokenizer.eos_token_id
model = AutoModelForCausalLM.from_pretrained(
    model, device_map="auto", 
    torch_dtype=torch.bfloat16,
    quantization_config=BitsAndBytesConfig
        (
            load_in_4bit=True, 
            bnb_4bit_compute_dtype=torch.bfloat16, 
            bnb_4bit_quant_type='nf4',
            bnb_4bit_use_double_quant=False 
        )
    )
model.enable_input_require_grads()

2026-03-06 09:12:56,599 - modelscope - INFO - Target directory already exists, skipping creation.


2026-03-06 09:12:57,433 - modelscope - INFO - Target directory already exists, skipping creation.
`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 4/4 [00:17<00:00,  4.31s/it]


## 3. Data Preprocess

In [ ]:
# <|im_start|>system
# You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
# <|im_start|>user
# What is 2 + 2?<|im_end|>
# <|im_start|>assistant

def process_func(example):
    """
    Convert dataset from a json file to a format that is acceptable to model.
    """

    MAX_LENGTH = 512 
    input_ids, attention_mask, labels = [], [], []
    system_prompt = SYSTEM_PROMPT.format(entities=entities_label)

    instruction = tokenizer(
        f"<|im_start|>system\n{system_prompt}<|im_end|>\n<|im_start|>user\n{example['sentence']}<|im_end|>\n<|im_start|>assistant\n",
        add_special_tokens=False
    )
    response = tokenizer(f"{json.dumps(example['entity_info'], ensure_ascii=False)}", add_special_tokens=False)

    input_ids = instruction['input_ids'] + response['input_ids'] + [tokenizer.eos_token_id]
    attention_mask = instruction['attention_mask'] + response['attention_mask'] + [1]
    labels = [-100] * len(instruction['input_ids']) + response['input_ids'] + [tokenizer.eos_token_id]
    if len(input_ids) > MAX_LENGTH:
        input_ids = input_ids[:MAX_LENGTH]
        attention_mask = attention_mask[:MAX_LENGTH]
        labels = labels[:MAX_LENGTH]
    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels                                                   
            }

train_dataset = train_dataset.map(process_func, remove_columns=train_dataset.column_names)
# test_dataset = test_dataset.map(process_func, remove_columns=test_dataset.column_names)
val_dataset = val_dataset.map(process_func, remove_columns=val_dataset.column_names)

Map: 100%|██████████| 657/657 [00:01<00:00, 364.84 examples/s]


## 4. Setup Lora 

In [7]:
for name, parameter in model.named_parameters():
    print(name)

model.embed_tokens.weight
model.layers.0.self_attn.q_proj.weight
model.layers.0.self_attn.q_proj.bias
model.layers.0.self_attn.k_proj.weight
model.layers.0.self_attn.k_proj.bias
model.layers.0.self_attn.v_proj.weight
model.layers.0.self_attn.v_proj.bias
model.layers.0.self_attn.o_proj.weight
model.layers.0.mlp.gate_proj.weight
model.layers.0.mlp.up_proj.weight
model.layers.0.mlp.down_proj.weight
model.layers.0.input_layernorm.weight
model.layers.0.post_attention_layernorm.weight
model.layers.1.self_attn.q_proj.weight
model.layers.1.self_attn.q_proj.bias
model.layers.1.self_attn.k_proj.weight
model.layers.1.self_attn.k_proj.bias
model.layers.1.self_attn.v_proj.weight
model.layers.1.self_attn.v_proj.bias
model.layers.1.self_attn.o_proj.weight
model.layers.1.mlp.gate_proj.weight
model.layers.1.mlp.up_proj.weight
model.layers.1.mlp.down_proj.weight
model.layers.1.input_layernorm.weight
model.layers.1.post_attention_layernorm.weight
model.layers.2.self_attn.q_proj.weight
model.layers.2.self

In [8]:
config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none"
)

## 5. Training 

In [9]:
model = get_peft_model(model, config)

args = TrainingArguments(
    output_dir="./output/Qwen2.5-7B",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    logging_steps=10,
    num_train_epochs=2,
    save_steps=100,
    learning_rate=1e-4,
    save_on_each_node=True,
    gradient_checkpointing=True,
    report_to="none",
    eval_strategy="steps",
    eval_steps=50,    
)

In [10]:
trainer = Trainer(
    model=model, 
    args=args,
    train_dataset=train_dataset, 
    eval_dataset=val_dataset,
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, padding=True),
    callbacks=[WandbCallback()]
)

trainer.train()

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Step,Training Loss,Validation Loss
50,0.091000,0.086048
100,0.079700,0.075855
150,0.070200,0.069985
200,0.065800,0.066498
250,0.068400,0.063538
300,0.057200,0.060820
350,0.057400,0.059762
400,0.039200,0.060062
450,0.041900,0.058223
500,0.039400,0.057159


TrainOutput(global_step=658, training_loss=0.0612182308747051, metrics={'train_runtime': 2280.0988, 'train_samples_per_second': 4.612, 'train_steps_per_second': 0.289, 'total_flos': 1.3574981637671731e+17, 'train_loss': 0.0612182308747051, 'epoch': 2.0})

In [11]:
wandb.finish()

wandb: ERROR The nbformat package was not found. It is required to save notebook history.


eval/loss,█▆▄▄▃▂▂▂▂▂▁▁▁
eval/runtime,█▃▁▂▂▄▃▃▄▂▃▁▂
eval/samples_per_second,▁▆█▇▇▅▆▆▅▇▆█▇
eval/steps_per_second,▁▆█▇▇▅▆▆▅▇▆█▇
train/epoch,▁▁▁▁▂▂▂▂▃▃▃▃▃▃▃▃▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▆▆▇▇▇███
train/global_step,▁▁▁▂▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇▇███
train/grad_norm,▁▂▂█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▂▁▁▁▁
train/learning_rate,████▇▇▇▇▇▇▆▆▆▆▆▅▅▅▅▅▅▅▅▄▄▄▄▃▃▃▃▃▃▃▂▂▂▁▁▁
train/loss,█▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁
eval/loss,0.05429
eval/runtime,34.919


## 6. Evaluation 

In [6]:
model = AutoModelForCausalLM.from_pretrained(
    "./qwen/Qwen/Qwen2___5-7B", device_map="auto", 
    torch_dtype=torch.bfloat16,
    quantization_config=BitsAndBytesConfig
        (
            load_in_4bit=True, 
            bnb_4bit_compute_dtype=torch.bfloat16, 
            bnb_4bit_quant_type='nf4',
            bnb_4bit_use_double_quant=False 
        )
    )
model_id = "./output/Qwen2.5-7B/checkpoint-658"
p_model = PeftModel.from_pretrained(model, model_id=model_id)
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-7B", use_fast=False, trust_remote_code=False)

`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 4/4 [00:16<00:00,  4.07s/it]


2026-03-09 16:20:07,118 - modelscope - INFO - Target directory already exists, skipping creation.


In [9]:
def calculate_f1(preds, truths):
    """
    preds: List of List of dicts. e.g. [[{"entity_text": the_substring_1, "entity_label": the_corresponding_label_1 }, ...], ...]
    truths: List of List of dicts. e.g. [[{"entity_text": the_substring_1, "entity_label": the_corresponding_label_1 }, ...], ...]
    """
    metrics = defaultdict(lambda: {'tp':0, 'fp':0, 'fn':0})

    for pred_list, gt_list in zip(preds, truths):

        def to_tuples(data):
            return [
                (x.get("entity_label"), x.get("entity_text"))
                for x in data
                if isinstance(x, dict)
            ]
        
        pred_tuples = to_tuples(pred_list)
        gt_tuples = to_tuples(gt_list)

        pred_counts = Counter(pred_tuples)
        gt_counts = Counter(gt_tuples)

        all_labels = set([x[0] for x in pred_counts.keys()] + [x[0] for x in gt_counts.keys()])

        for label in all_labels:
            # Filter counts for this specific label
            p_counts = {k[1]: v for k, v in pred_counts.items() if k[0] == label}
            g_counts = {k[1]: v for k, v in gt_counts.items() if k[0] == label}
            
            # Intersection (True Positives)
            # If pred has 2 "Apple" and truth has 3 "Apple", TP is 2.
            tp = 0
            for text in p_counts:
                if text in g_counts:
                    tp += min(p_counts[text], g_counts[text])
            
            # False Positives: Total Predicted - TP
            total_pred = sum(p_counts.values())
            fp = total_pred - tp
            
            # False Negatives: Total Truth - TP
            total_truth = sum(g_counts.values())
            fn = total_truth - tp

            metrics[label]['tp'] += tp
            metrics[label]['fp'] += fp
            metrics[label]['fn'] += fn
    print(f"{'Label':<15} {'Prec':<10} {'Recall':<10} {'F1':<10}")
    total_tp = 0
    total_fp = 0 
    total_fn = 0

    for label, counts in metrics.items():
        tp, fp, fn = counts['tp'], counts['fp'], counts['fn']

        precision = tp / (tp + fp) if (tp + fp) != 0 else 0.0 
        recall = tp / (tp + fn) if (tp + fn) != 0 else 0.0 
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

        print(f"{label:<15} {precision:.4f}     {recall:.4f}     {f1:.4f}")

        total_tp += tp
        total_fp += fp 
        total_fn += fn

    micro_precision = total_tp / (total_tp + total_fp) if (total_tp + total_fp) != 0 else 0.0
    micro_recall = total_tp / (total_tp + total_fn) if (total_tp + total_fn) != 0 else 0.0 
    micro_f1 = 2 * micro_precision * micro_recall / (micro_precision + micro_recall) if (micro_precision + micro_recall) > 0 else 0.0
    print("-" * 45)
    print(f"{'Overall (Micro)':<15} {micro_precision:.4f}     {micro_recall:.4f}     {micro_f1:.4f}")

def predict_batch(batch_messages, model, tokenizer):
    # 1. Apply template to EACH conversation in the batch individually
    batch_texts = [
        tokenizer.apply_chat_template(msg, tokenize=False, add_generation_prompt=True)
        for msg in batch_messages
    ]
    
    # 2. Tokenize the list of strings (Padding is crucial here!)
    model_inputs = tokenizer(
        batch_texts, 
        return_tensors="pt", 
        padding=True, 
        truncation=True
    ).to(model.device)
    
    # 3. Generate
    generated_ids = model.generate(**model_inputs, max_new_tokens=512)
    
    # 4. Slice off input tokens
    input_lengths = [len(x) for x in model_inputs.input_ids]
    final_ids = [
        gen_id[in_len:] for gen_id, in_len in zip(generated_ids, input_lengths)
    ]

    # 5. Decode
    responses = tokenizer.batch_decode(final_ids, skip_special_tokens=True)
    return responses
    
# Ensure padding is on the left for generation
tokenizer.padding_side = "left" 
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

preds = []
truths = []
batch_size = 4
for start in tqdm(range(0, len(val_dataset), batch_size)):
    end = min(start + batch_size, len(val_dataset))
    batch_data = val_dataset[start:end]
    batch_messages = []

    if isinstance(batch_data, dict):
        # Convert {'sentence': ['a', 'b'], ...} to [{'sentence': 'a', ...}, ...]
        rows = [
            dict(zip(batch_data, t))  # dict() takes those paired tuples and turns them into a standard Python dictionary
            for t in zip(*batch_data.values()) # t = (sentence, entitiy_info)
        ]
    else:
        rows = batch_data

    for row in rows:
        sentence = row["sentence"]
        entity_info = row["entity_info"]
        
        # Store Truth
        truths.append(entity_info)

        # Prepare Message
        messages = [ 
            {"role": "system", "content": f"{SYSTEM_PROMPT.format(entities=entities_label)}"},
            {"role": "user", "content": f"{sentence}"},
        ]
        batch_messages.append(messages)

    # Predict
    responses = predict_batch(batch_messages, p_model, tokenizer)

    print(f"{batch_messages[0][0]}\n\n{batch_messages[0][1]}\n\n{responses[0]}") # show one example in the batch
    for response in responses:
        try:
            parsed = json.loads(response)

            if isinstance(parsed, dict) and "entity_info" in parsed:
                preds.append(parsed["entity_info"])
            elif isinstance(parsed, list):
                preds.append(parsed)
            else:
                preds.append([])
        except json.JSONDecodeError as e:
            preds.append([])
            print(f"{e}: {responses}")

  1%|          | 1/164 [00:15<42:23, 15.60s/it]Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


{'role': 'system', 'content': '\nYou are an expert in the field of Named Entity Recognition (NER). You need to extract the entities from the given sentence.\nThe possible entities are [\'中医证候\', \'中医治疗\', \'其他治疗\', \'临床表现\', \'西医诊断\', \'方剂\', \'中医治则\', \'中医诊断\', \'西医治疗\', \'中药\'].\n\nOutput in the following JSON format:\n    {\n        "entity_info": [\n            {\n                "entity_text": the_substring_1,\n                "entity_label": the_corresponding_label_1         \n            },\n            {\n                "entity_text": the_substring_2,\n                "entity_label": the_corresponding_label_2         \n            },\n            ...\n        ]\n    }\n\nNotes:\n\nEach line of output must be a valid JSON string.\n\n'}

{'role': 'user', 'content': '投活络效灵丹加味：当归、丹参各１５ｇ，生乳香、生没药各６ｇ，柴胡１２ｇ，白芍、黄芩、大黄各１０ｇ，蒲公英３０ｇ，甘草５ｇ'}

[{"entity_text": "活络效灵丹加味", "entity_label": "方剂"}, {"entity_text": "当归", "entity_label": "中药"}, {"entity_text": "丹参", "entity_label": "中药"}, {"entity_te

  1%|          | 2/164 [00:19<23:39,  8.76s/it]Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


{'role': 'system', 'content': '\nYou are an expert in the field of Named Entity Recognition (NER). You need to extract the entities from the given sentence.\nThe possible entities are [\'中医证候\', \'中医治疗\', \'其他治疗\', \'临床表现\', \'西医诊断\', \'方剂\', \'中医治则\', \'中医诊断\', \'西医治疗\', \'中药\'].\n\nOutput in the following JSON format:\n    {\n        "entity_info": [\n            {\n                "entity_text": the_substring_1,\n                "entity_label": the_corresponding_label_1         \n            },\n            {\n                "entity_text": the_substring_2,\n                "entity_label": the_corresponding_label_2         \n            },\n            ...\n        ]\n    }\n\nNotes:\n\nEach line of output must be a valid JSON string.\n\n'}

{'role': 'user', 'content': '结论：辨证针刺治疗ＦＤ能明显改善患者的生活质量，为治疗ＦＤ的有效方法'}

[{"entity_text": "辨证针刺", "entity_label": "中医治疗"}]


  2%|▏         | 3/164 [00:23<17:38,  6.57s/it]Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


...

{'role': 'system', 'content': '\nYou are an expert in the field of Named Entity Recognition (NER). You need to extract the entities from the given sentence.\nThe possible entities are [\'中医证候\', \'中医治疗\', \'其他治疗\', \'临床表现\', \'西医诊断\', \'方剂\', \'中医治则\', \'中医诊断\', \'西医治疗\', \'中药\'].\n\nOutput in the following JSON format:\n    {\n        "entity_info": [\n            {\n                "entity_text": the_substring_1,\n                "entity_label": the_corresponding_label_1         \n            },\n            {\n                "entity_text": the_substring_2,\n                "entity_label": the_corresponding_label_2         \n            },\n            ...\n        ]\n    }\n\nNotes:\n\nEach line of output must be a valid JSON string.\n\n'}

{'role': 'user', 'content': '胃食管反流性咳嗽；通降和胃方；奥美拉唑；脏腑相关理论'}

[{"entity_text": "胃食管反流性咳嗽", "entity_label": "西医诊断"}, {"entity_text": "奥美拉唑", "entity_label": "西医治疗"}]


 99%|█████████▉| 163/164 [25:28<00:07,  7.74s/it]Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


{'role': 'system', 'content': '\nYou are an expert in the field of Named Entity Recognition (NER). You need to extract the entities from the given sentence.\nThe possible entities are [\'中医证候\', \'中医治疗\', \'其他治疗\', \'临床表现\', \'西医诊断\', \'方剂\', \'中医治则\', \'中医诊断\', \'西医治疗\', \'中药\'].\n\nOutput in the following JSON format:\n    {\n        "entity_info": [\n            {\n                "entity_text": the_substring_1,\n                "entity_label": the_corresponding_label_1         \n            },\n            {\n                "entity_text": the_substring_2,\n                "entity_label": the_corresponding_label_2         \n            },\n            ...\n        ]\n    }\n\nNotes:\n\nEach line of output must be a valid JSON string.\n\n'}

{'role': 'user', 'content': '肠宁汤治疗肠易激综合征５３例疗效观察'}

[{"entity_text": "肠宁汤", "entity_label": "方剂"}, {"entity_text": "肠易激综合征", "entity_label": "西医诊断"}]


100%|██████████| 164/164 [25:34<00:00,  9.36s/it]

{'role': 'system', 'content': '\nYou are an expert in the field of Named Entity Recognition (NER). You need to extract the entities from the given sentence.\nThe possible entities are [\'中医证候\', \'中医治疗\', \'其他治疗\', \'临床表现\', \'西医诊断\', \'方剂\', \'中医治则\', \'中医诊断\', \'西医治疗\', \'中药\'].\n\nOutput in the following JSON format:\n    {\n        "entity_info": [\n            {\n                "entity_text": the_substring_1,\n                "entity_label": the_corresponding_label_1         \n            },\n            {\n                "entity_text": the_substring_2,\n                "entity_label": the_corresponding_label_2         \n            },\n            ...\n        ]\n    }\n\nNotes:\n\nEach line of output must be a valid JSON string.\n\n'}

{'role': 'user', 'content': '目的：观察针刺联合穴位贴敷治疗脾胃虚弱型慢性萎缩性胃炎（ＣＡＧ）的临床效果'}

[{"entity_text": "针刺", "entity_label": "中医治疗"}, {"entity_text": "脾胃虚弱", "entity_label": "中医证候"}, {"entity_text": "慢性萎缩性胃炎", "entity_label": "西医诊断"}]


In [10]:
calculate_f1(preds, truths)

Label           Prec       Recall     F1        
方剂              0.5188     0.5750     0.5455
中药              0.7281     0.9114     0.8095
西医诊断            0.8563     0.9511     0.9012
中医证候            0.7235     0.9044     0.8039
中医治疗            0.7982     0.8198     0.8089
其他治疗            0.1667     0.1250     0.1429
临床表现            0.6070     0.7073     0.6533
中医治则            0.5833     0.3962     0.4719
西医治疗            0.6667     0.6818     0.6742
中医诊断            0.7045     0.7381     0.7209
---------------------------------------------
Overall (Micro) 0.7058     0.8091     0.7540
